In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
from typing import Hashable, Any

import pandas as pd
import openai  

import os
import cohere

co_client = cohere.Client()


### Read the sampled dataset woth amazon inventory data

In [2]:
df_items = pd.read_json('../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl', lines=True)

In [3]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,All Electronics,Syncwire Bluetooth 5.3 FM Transmitter Car Adap...,4.6,188,[[𝙃𝙪𝙢𝙖𝙣𝙞𝙯𝙚𝙙 𝘼𝙢𝙗𝙞𝙚𝙣𝙩 𝙇𝙞𝙜𝙝𝙩 𝙎𝙬𝙞𝙩𝙘𝙝 𝘿𝙚𝙨𝙞𝙜𝙣] Doubl...,[],24.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Syncwire Bluetooth 5.3 FM Transmit...,Syncwire,"[Electronics, Portable Audio & Video, MP3 & MP...",{'Package Dimensions': '3.46 x 2.36 x 1.65 inc...,B0BTP3PQ6Y,NaN
1,Cell Phones & Accessories,Google Pixel 7 Pro - 5G Android Phone - Unlock...,4.2,1385,[Google Pixel 7 Pro is Google’s best-of-everyt...,"[Meet Pixel 7 Pro, Google’s best-of-everything...",1099.00,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Best camera feature on Google Pixe...,Google,"[Electronics, New Arrivals - Electronics]",{'Product Dimensions': '6.42 x 3.02 x 0.34 inc...,B0BVX6BBMJ,NaN
2,Cell Phones & Accessories,Shangpule Compatible with Fitbit Charge 5 Band...,3.8,315,[Compatible with Fitbit Charge 5 Fitness Activ...,[],16.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Would not stay attached', 'url': '...",NaN,"[Electronics, Wearable Technology, Arm & Wrist...","{'Date First Available': 'October 14, 2022'}",B0BJ6PY8Z6,NaN
3,All Electronics,NexFoto Digital Picture Frame with Remote Cont...,4.3,114,[Every Moment Counts: Sharing life moments wit...,[],59.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Every Moment Counts.', 'url': 'htt...",NexFoto,"[Electronics, Camera & Photo, Lighting & Studi...",{'Product Dimensions': '8.7 x 6.14 x 0.98 inch...,B0C2TSD8FK,NaN
4,All Electronics,"ivoler Laptop Stand, Laptop Holder Riser Compu...",4.7,6783,[COMPATIBILITY: The laptop & tablet stand supp...,[],15.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'laptop stand with must have featur...,ivoler,"[Electronics, Computers & Accessories, Laptop ...",{'Product Dimensions': '9.4 x 1.73 x 1 inches'...,B0B63VL3N6,NaN


In [4]:
list[tuple[Hashable, Any]](df_items["features"].items())[1] ## feature 0 is null

(1,
 ['Google Pixel 7 Pro is Google’s best-of-everything phone; powered by Google Tensor G2, it’s faster, more efficient, and more secure, with the best photo and video quality yet on Pixel[1].Other camera description:Front,Rear',
  'Unlocked Android 5G phone gives you the flexibility to change carriers and choose your own data plan[2]; works with Google Fi, Verizon, T-Mobile, AT&T, and other major carriers',
  'Pixel’s Adaptive Battery can last over 24 hours; when Extreme Battery Saver is turned on, it can last up to 72 hours[3]',
  'The Google Pixel 7 Pro 6.7-inch Smooth Display makes everything stunning and immersive[4]; it intelligently adjusts up to 120Hz for smoother, more responsive performance[4]',
  'Google Pixel 7 Pro has a 5x telephoto lens with 30x Super Res Zoom[5]; the upgraded ultrawide lens powers Macro Focus to capture the smallest details',
  'With Google Tensor G2 and the Titan M2 security chip, your Pixel is built with multiple layers of security to help keep your p

In [5]:
list[tuple[Hashable, Any]](df_items["images"].items())[0]

(0,
 [{'thumb': 'https://m.media-amazon.com/images/I/41nIhnMtDIL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41nIhnMtDIL._AC_.jpg',
   'variant': 'MAIN',
   'hi_res': 'https://m.media-amazon.com/images/I/71-AHxfs0BL._AC_SL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/51W+VnyLGaL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51W+VnyLGaL._AC_.jpg',
   'variant': 'PT01',
   'hi_res': 'https://m.media-amazon.com/images/I/811qrQPq0lL._AC_SL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/51y374XGX4L._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51y374XGX4L._AC_.jpg',
   'variant': 'PT02',
   'hi_res': 'https://m.media-amazon.com/images/I/71gDXqGU1NL._AC_SL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/51EUA-oQ-NL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51EUA-oQ-NL._AC_.jpg',
   'variant': 'PT03',
   'hi_res': 'https://m.media-amazon.com/images/I/71RqqWz0l7L._AC_SL1

## Preprocess title and feature 

In [6]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [7]:
def extract_first_large_image(row):
    return row["images"][0].get("large", "")

In [8]:
df_items["description"] = df_items.apply(preprocess_description, axis=1)
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)

In [9]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,image
0,All Electronics,Syncwire Bluetooth 5.3 FM Transmitter Car Adap...,4.6,188,[[𝙃𝙪𝙢𝙖𝙣𝙞𝙯𝙚𝙙 𝘼𝙢𝙗𝙞𝙚𝙣𝙩 𝙇𝙞𝙜𝙝𝙩 𝙎𝙬𝙞𝙩𝙘𝙝 𝘿𝙚𝙨𝙞𝙜𝙣] Doubl...,Syncwire Bluetooth 5.3 FM Transmitter Car Adap...,24.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Syncwire Bluetooth 5.3 FM Transmit...,Syncwire,"[Electronics, Portable Audio & Video, MP3 & MP...",{'Package Dimensions': '3.46 x 2.36 x 1.65 inc...,B0BTP3PQ6Y,NaN,https://m.media-amazon.com/images/I/41nIhnMtDI...
1,Cell Phones & Accessories,Google Pixel 7 Pro - 5G Android Phone - Unlock...,4.2,1385,[Google Pixel 7 Pro is Google’s best-of-everyt...,Google Pixel 7 Pro - 5G Android Phone - Unlock...,1099.00,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Best camera feature on Google Pixe...,Google,"[Electronics, New Arrivals - Electronics]",{'Product Dimensions': '6.42 x 3.02 x 0.34 inc...,B0BVX6BBMJ,NaN,https://m.media-amazon.com/images/I/4161bAE+CA...
2,Cell Phones & Accessories,Shangpule Compatible with Fitbit Charge 5 Band...,3.8,315,[Compatible with Fitbit Charge 5 Fitness Activ...,Shangpule Compatible with Fitbit Charge 5 Band...,16.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Would not stay attached', 'url': '...",NaN,"[Electronics, Wearable Technology, Arm & Wrist...","{'Date First Available': 'October 14, 2022'}",B0BJ6PY8Z6,NaN,https://m.media-amazon.com/images/I/413TlX98O-...
3,All Electronics,NexFoto Digital Picture Frame with Remote Cont...,4.3,114,[Every Moment Counts: Sharing life moments wit...,NexFoto Digital Picture Frame with Remote Cont...,59.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Every Moment Counts.', 'url': 'htt...",NexFoto,"[Electronics, Camera & Photo, Lighting & Studi...",{'Product Dimensions': '8.7 x 6.14 x 0.98 inch...,B0C2TSD8FK,NaN,https://m.media-amazon.com/images/I/4153deE+9c...
4,All Electronics,"ivoler Laptop Stand, Laptop Holder Riser Compu...",4.7,6783,[COMPATIBILITY: The laptop & tablet stand supp...,"ivoler Laptop Stand, Laptop Holder Riser Compu...",15.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'laptop stand with must have featur...,ivoler,"[Electronics, Computers & Accessories, Laptop ...",{'Product Dimensions': '9.4 x 1.73 x 1 inches'...,B0B63VL3N6,NaN,https://m.media-amazon.com/images/I/41WRzASGgj...


### Look the "title" and "features" it have the same description so we use .join() to combine it with each other on one string line on index 0

In [10]:
list[tuple[Hashable, Any]](df_items["description"].items())[0]

(0,
 'Syncwire Bluetooth 5.3 FM Transmitter Car Adapter 48W (PD 36W & 12W) [Light Switch] [HiFi Bass Sound] [Fast Charging] Wireless Radio Music Adapter LED Display Hands-Free Calling Support USB Drive [𝙃𝙪𝙢𝙖𝙣𝙞𝙯𝙚𝙙 𝘼𝙢𝙗𝙞𝙚𝙣𝙩 𝙇𝙞𝙜𝙝𝙩 𝙎𝙬𝙞𝙩𝙘𝙝 𝘿𝙚𝙨𝙞𝙜𝙣] Double press the red "B" button to turn off/on the colored lights directly. [𝙐𝙥𝙜𝙧𝙖𝙙𝙚𝙙 𝘽𝙡𝙪𝙚𝙩𝙤𝙤𝙩𝙝 𝟱.𝟯 𝙏𝙚𝙘𝙝𝙣𝙤𝙡𝙤𝙜𝙮] The latest Bluetooth 5.3 technology, optimizing the transmission delay and connection defect problems, Bluetooth 5.3 will bring you a faster, more stable and smarter connection with less sound quality loss,uninterrupted calls and music experience. It takes only 1 second to complete pairing, and no need to repeat pairing after the first successful pairing with your device. [𝙄𝙣𝙩𝙚𝙡𝙡𝙞𝙜𝙚𝙣𝙩 𝘿𝙪𝙖𝙡 𝙐𝙎𝘽 𝙋𝙤𝙧𝙩𝙨 & 𝙁𝙖𝙨𝙩 𝘾𝙝𝙖𝙧𝙜𝙞𝙣𝙜] Coming with 2 USB ports(PD 36W and USB A 12W), can charge two devices simultaneously. Built-in over-current protection, over-voltage protection, intelligent temperature control, short circuit protection, it will effectively en

## Sample 50 items from the dataset

In [11]:
df_sample = df_items.sample(50, random_state=42)

In [12]:
len(df_sample)

50

we prepare 50 sample with these below data feature for embeding 

In [13]:
data_to_embed = df_sample[["description", "image", "rating_number", "price", "average_rating", "parent_asin"]]. to_dict(orient="records")

In [14]:
data_to_embed

[{'description': "BoYata Screen Cleaner Kit 4PCS, 3-in-1 Touchscreen Mist Cleaner Spray Bottle Tool & Microfiber Cloth, Portable Reusable Fingerprint-Proof Spray Bottle for Phone/Laptop/Tablet/TV/Monitor/Car Screens Not Containing Cleaning Fluid: BoYata all-in-one detachable fingerprint-proof computer screen cleaner consists of a protective case, microfiber cloth shell and spray bottle (without cleaning fluid), integrated spray and wipe in one. Extremely convenient to use after filling cleaning fluid. Note: please don't put corrosive liquids. Wide Compatibility: The touch screen mist cleaner is useful for screen cleaning. Suitable for all phones/ tablets/ laptops/ monitors/ TVs/ cars screens, glass surface etc. Spray and wipe in one for quick and easy cleaning, making your screen look as good as new. Safe and no damage to your screen. Powerful Cleaning: The updated microfiber cloth of the screen cleaner is easy to move fingerprints, dust, grease, dirt on screen without streaking or scr

### Define the embedding function 

In [16]:
co_client = cohere.Client(api_key=os.environ.get("COHERE_API_KEY"))
# Test the Cohere embedding model
response = co_client.embed(
    texts=["Random text"],
    model="embed-english-v3.0",
    input_type="search_document"
)
# Print the first few numbers to verify it worked
print(response.embeddings[0][:5])

[-0.014183044, 0.0037727356, 0.005558014, -0.017471313, -0.037322998]


In [18]:
len(response.embeddings[0])

1024

In [19]:
# The Cohere version of your teacher's function
def get_embedding(text, model="embed-english-v3.0"):
    response = co_client.embed(
        texts=[text],
        model=model,
        input_type="search_query"
    )
    return response.embeddings[0]
get_embedding("Hello world!")

[-0.03829956,
 -0.047210693,
 -0.08502197,
 -0.08239746,
 -0.008659363,
 0.0009498596,
 -0.04425049,
 0.066467285,
 0.011169434,
 0.014572144,
 -0.030670166,
 0.03543091,
 0.049072266,
 -0.04724121,
 0.051757812,
 -0.028213501,
 0.0491333,
 0.007873535,
 -0.010040283,
 -0.009513855,
 -0.019241333,
 0.014251709,
 0.011474609,
 0.017791748,
 0.044952393,
 -0.008796692,
 0.004055023,
 -0.0014390945,
 -0.058685303,
 -0.043060303,
 -0.007385254,
 0.060424805,
 0.019363403,
 0.024154663,
 -0.009483337,
 0.037261963,
 -0.018676758,
 0.006378174,
 0.010848999,
 -0.0011062622,
 0.015357971,
 -0.0018415451,
 0.012664795,
 -0.039855957,
 -0.030059814,
 0.0019159317,
 -0.040924072,
 -0.0104522705,
 0.03942871,
 -0.029159546,
 0.007686615,
 -0.012924194,
 -0.00027561188,
 0.009880066,
 -0.0062942505,
 0.007118225,
 -0.046142578,
 -0.009513855,
 0.009811401,
 0.07287598,
 0.025146484,
 0.0074005127,
 0.003440857,
 -0.013320923,
 0.010414124,
 0.0021324158,
 0.013366699,
 0.011276245,
 0.0289917,
 0.

In [20]:
get_embedding("Hello world!")

[-0.03829956,
 -0.047210693,
 -0.08502197,
 -0.08239746,
 -0.008659363,
 0.0009498596,
 -0.04425049,
 0.066467285,
 0.011169434,
 0.014572144,
 -0.030670166,
 0.03543091,
 0.049072266,
 -0.04724121,
 0.051757812,
 -0.028213501,
 0.0491333,
 0.007873535,
 -0.010040283,
 -0.009513855,
 -0.019241333,
 0.014251709,
 0.011474609,
 0.017791748,
 0.044952393,
 -0.008796692,
 0.004055023,
 -0.0014390945,
 -0.058685303,
 -0.043060303,
 -0.007385254,
 0.060424805,
 0.019363403,
 0.024154663,
 -0.009483337,
 0.037261963,
 -0.018676758,
 0.006378174,
 0.010848999,
 -0.0011062622,
 0.015357971,
 -0.0018415451,
 0.012664795,
 -0.039855957,
 -0.030059814,
 0.0019159317,
 -0.040924072,
 -0.0104522705,
 0.03942871,
 -0.029159546,
 0.007686615,
 -0.012924194,
 -0.00027561188,
 0.009880066,
 -0.0062942505,
 0.007118225,
 -0.046142578,
 -0.009513855,
 0.009811401,
 0.07287598,
 0.025146484,
 0.0074005127,
 0.003440857,
 -0.013320923,
 0.010414124,
 0.0021324158,
 0.013366699,
 0.011276245,
 0.0289917,
 0.

## Create Qdrant collection

In [21]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [22]:
qdrant_client.recreate_collection(
    collection_name="amazon-items-collection-00",
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
)

C:\Users\Chanreach\AppData\Local\Temp\ipykernel_30044\3101124450.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant_client.recreate_collection(


True

## Embed data

Test

In [23]:
pointstruct = PointStruct(
    id=0,
    vector=get_embedding("Test text"),
    payload={
        "text": "Test text",
        "model": "embed-english-v3.0", # <-- Make sure you use the Gemini model here!
    },
)

In [24]:
pointstruct

PointStruct(id=0, vector=[-0.046691895, -0.028549194, -0.022262573, -0.012138367, -0.053253174, 0.005973816, -0.03729248, 0.035095215, 0.00072717667, 0.056549072, -0.058898926, 0.04711914, -0.0037879944, 0.030426025, -0.008743286, -0.031097412, 0.027297974, 0.003622055, -0.005569458, -0.008804321, 0.011695862, 0.024276733, -0.033325195, -0.024246216, 0.006603241, 0.036895752, 0.02720642, -0.0031814575, -0.034210205, -0.031051636, 0.013519287, 0.005214691, 0.018844604, 0.025131226, -0.009140015, 0.026016235, -0.0126571655, -0.003353119, -0.0034751892, 0.01687622, 0.01436615, 0.022659302, -0.0071144104, 0.008277893, -0.013969421, -0.06896973, 0.046142578, -0.02116394, 0.048980713, 0.0063285828, -0.0016746521, 0.041748047, -0.0044517517, -0.009796143, -0.017807007, -0.0019083023, -0.07128906, 0.049743652, 0.015594482, 0.01902771, 0.009994507, 0.036834717, 0.022323608, 0.0031471252, 0.039031982, -0.025543213, 0.036712646, 0.03062439, 0.033935547, 0.0137786865, -0.016677856, 0.0099487305, 0

## Amazon Data

In [25]:
pointstructs = []
for i, data in enumerate(data_to_embed):
    embediing = get_embedding(data["description"])
    pointstructs.append(
        PointStruct(
            id=i,
            vector=embediing,
            payload=data,
        )
    )

In [26]:
pointstructs

[PointStruct(id=0, vector=[0.010787964, -0.026931763, -0.00573349, -0.11657715, -0.02166748, 0.035003662, -0.0791626, 0.03744507, -0.01096344, 0.013832092, -0.02557373, 0.022872925, -0.024307251, -0.011390686, 0.038024902, -0.030151367, 0.030532837, -0.019973755, 0.011116028, -0.031280518, -0.036224365, -0.0075035095, 0.019119263, -0.05718994, -0.036468506, 0.0736084, -0.03640747, -0.023071289, -0.0496521, -0.020736694, -0.008399963, 0.08496094, -0.009918213, 0.06112671, -0.06945801, -0.016326904, -0.016815186, 0.031173706, 0.0038967133, 0.01184082, -0.00289917, -0.012809753, -0.013931274, -0.047821045, -0.03930664, 0.022613525, -0.014030457, 0.003791809, 0.0036201477, -0.021209717, -0.017456055, 0.02078247, -0.025985718, 0.0126953125, 0.014282227, 0.041259766, -0.022476196, -0.03112793, -0.0053977966, 0.06573486, 0.039215088, 0.0043029785, -0.00040364265, -0.04925537, -0.007835388, -0.011138916, -0.044952393, 0.0026435852, 0.052337646, 0.027069092, -0.0061454773, -0.015670776, -0.0213

In [27]:
len(pointstructs)

50

## Write embedded data to Qdrant database

In [28]:
qdrant_client.upsert(
    collection_name="amazon-items-collection-00",
    wait=True,
    points=pointstructs,
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

## Define a fucntion for data retrieval

In [29]:
def retrieve_data(query, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="amazon-items-collection-00",
        query=query_embedding,
        limit=k,
    )
    return results

### Test Retrieval

In [30]:
retrieve_data("what kind of products that are available?", k=10).points

[ScoredPoint(id=48, version=1, score=0.326369, payload={'description': "Parblo PR114 Drawing Tablet Stand Holder Adjustable,Sturdy Portable Monitor Stand Desktop Stand Compatible with ipad/air/Mini/pro,Kindle,Surface,Samsung Tab,Phone Foldable & Wide Compatibility: With Foldable design,weight is only 200g,Our tablet holder is convenient to storage and transportation.Convenient to use at home or in conference rooms, anywhere outdoors.The Tablet stand for desk can fit for up to 11inch tablet,Compatible for iPad mini /air/pro series,Samsung Galaxy Series,Huawei MatePad Series, Lenovo Tab P11 Pro,Microsoft Surface,All Phones. Exquisite structure & weight resistance 10kg: From the aluminum alloy material to the overall structure, the tablet holder details are carefully crafted. Unique double triangular structure, no tilt, can be used stably outdoors, and can withstand 10 kg. Adjustable angle & improve posture: The ipad holder stand is Based on ergonomics, it can adjust the most suitable ang